In [1]:
import os

In [2]:
%pwd

'c:\\Users\\User\\Desktop\\DS\\Kidney_cancer\\research'

In [3]:
os.chdir('..')

In [4]:
%pwd

'c:\\Users\\User\\Desktop\\DS\\Kidney_cancer'

In [5]:
import dagshub
dagshub.init(repo_owner='SalazarV4', repo_name='Kidney_cancer', mlflow=True)

Accessing as SalazarV4

Initialized MLflow to track repo "SalazarV4/Kidney_cancer"

Repository SalazarV4/Kidney_cancer initialized!

In [6]:
import torch

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: int
    params_batch_size: int

In [8]:
from kidney_cancer.constants import  CONFIG_FILE_PATH, PARAMS_FILE_PATH
from kidney_cancer.utils.common import read_yaml, create_directories, save_json
import torch
from torch import nn

In [9]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_evaluation_config(self) -> EvaluationConfig:
        evaluation_config= EvaluationConfig(
            path_of_model=Path("artifacts/base_model/base_model_updated.pth"),
            training_data=Path("artifacts/data_ingestion/Kidney_dataset"),
            mlflow_uri="https://dagshub.com/SalazarV4/Kidney_cancer.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

        return evaluation_config
    



In [11]:
config = ConfigurationManager()
eval_config = config.get_evaluation_config()

[2025-07-30 11:11:40,655: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-30 11:11:40,658: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-30 11:11:40,660: INFO: common: created directory at: artifacts]


In [12]:
eval_config 

EvaluationConfig(path_of_model=WindowsPath('artifacts/base_model/base_model_updated.pth'), training_data=WindowsPath('artifacts/data_ingestion/Kidney_dataset'), all_params=ConfigBox({'AUGMENTATION': False, 'IMAGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 1, 'CLASSES': 2, 'WEIGHTS': 'IMAGENET1K_V1', 'LEARNING_RATE': 0.01}), mlflow_uri='https://dagshub.com/SalazarV4/Kidney_cancer.mlflow', params_image_size=BoxList([224, 224, 3]), params_batch_size=16)

In [13]:
import mlflow

In [15]:
print(mlflow.get_tracking_uri())

https://dagshub.com/SalazarV4/Kidney_cancer.mlflow


In [16]:
import torch
import mlflow
from urllib.parse import urlparse
from pathlib import Path
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from kidney_cancer import logger

In [19]:
urlparse(mlflow.get_tracking_uri())

ParseResult(scheme='https', netloc='dagshub.com', path='/SalazarV4/Kidney_cancer.mlflow', params='', query='', fragment='')

In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.loss_fn = nn.CrossEntropyLoss()

    def test_loader(self):
        self.transforms = v2.Compose([
            v2.ToImage(),
            v2.Resize(size=[224,224]),
            v2.ToDtype(dtype=torch.float32,scale=True)])
    
        self.test_data = ImageFolder(
            root=self.config.training_data / "Test",
            transform=self.transforms)
        
        self.test_dataloader = DataLoader(self.test_data,
                                           batch_size=self.config.params_batch_size,
                                           shuffle=True)
    
    def test(self):
        self.model.eval() 

        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            for batch, (X_test, y_test) in enumerate(self.test_dataloader):
                print(f"Batch number {batch+1}")
                X_test, y_test = X_test.to(self.device), y_test.to(self.device)

                test_pred_logits = self.model(X_test)

                loss = self.loss_fn(test_pred_logits, y_test)
                test_loss += loss.item()

                test_pred_labels = test_pred_logits.argmax(dim=1)

                test_acc += ((test_pred_labels == y_test).sum().item()/len(test_pred_labels))

            test_loss = test_loss / len(self.test_dataloader)
            test_acc = test_acc / len(self.test_dataloader)
            logger.info("Testing Loss: %s | Testing Accuracy: %s",test_loss,test_acc)
            
            return test_loss, test_acc
        

    def load_model(self):
        return torch.load(self.config.path_of_model, weights_only=False)
    
    def evaluation(self):
        self.model = self.load_model()
        self.test_loader()
        self.score = self.test()
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0],
                 "accuracy": self.score[1]}
            )

            if tracking_url_type_store != "file":
                mlflow.pytorch.log_model(pytorch_model=self.model,
                                         artifact_path="model",
                                         registered_model_name="VGG16")
            else:
                mlflow.pytorch.log_model(self.model,
                                         artifact_path="model")

In [12]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    
except Exception as e:
    raise e

[2025-07-30 09:31:52,469: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-30 09:31:52,482: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-30 09:31:52,484: INFO: common: created directory at: artifacts]
Batch number 1
Batch number 2
Batch number 3
Batch number 4
Batch number 5
Batch number 6
Batch number 7
Batch number 8
Batch number 9
Batch number 10
Batch number 11
Batch number 12
Batch number 13
Batch number 14
Batch number 15
Batch number 16
Batch number 17
Batch number 18
Batch number 19
Batch number 20
Batch number 21
Batch number 22
Batch number 23
Batch number 24
Batch number 25
Batch number 26
Batch number 27
Batch number 28
Batch number 29
[2025-07-30 09:33:33,563: INFO: 1349385742: Testing Loss: 0.6872785029740169 | Testing Accuracy: 0.6745689655172413]
[2025-07-30 09:33:33,570: INFO: common: json file saved at: scores.json]


2025/07/30 09:33:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run unequaled-seal-147 at: https://dagshub.com/SalazarV4/Kidney_cancer.mlflow/#/experiments/0/runs/0354e3c750034ee29b636b01d0d09a48
🧪 View experiment at: https://dagshub.com/SalazarV4/Kidney_cancer.mlflow/#/experiments/0


KeyboardInterrupt: 